#init 

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col,trim
from pyspark.sql.types import StringType

# Reading from Bronze
#Data Tranformation
#write into Silver Table



# Working with Customer Info Table 

In [0]:
df=spark.table("workspace.bronze.cust_info")





In [0]:
#Trimming the string for white Space
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))



In [0]:
#Renaiming Column        
df=df.withColumnRenamed("cst_id","customer_id")
df = df.withColumnRenamed("cst_key", "customer_key")
df = df.withColumnRenamed("cst_firstname", "customer_firstname")
df = df.withColumnRenamed("cst_lastname",   "customer_lastname")
df = df.withColumnRenamed("cst_marital_status", "customer_marital_status")
df = df.withColumnRenamed("cst_gndr", "customer_gender")
df = df.withColumnRenamed("cst_create_date", "customer_create_date")
df.display()

In [0]:
#Normalization
df=df.withColumn("customer_gender", F.when(col("customer_gender") == "M", "Male")
                                          .when(col("customer_gender") == "F", "Female")
                                          .otherwise("N/A"))
df=df.withColumn("customer_marital_status", F.when(col("customer_marital_status") == "S", "Single")
                                          .when(col("customer_marital_status") == "M", "Married")
                                          .otherwise("N/A"))

In [0]:
df.display()

In [0]:
%sql
select * from silver.sales_orders

In [0]:
df=spark.table("silver.cust_info")
df.describe().display()

#Writing to Silver layer
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.cust_info")
  )

# Sales info from LOC table 

In [0]:
df2=spark.table("workspace.bronze.loc_a101")

#change the data typr from string to date format by casting string to date
df2 = df2.withColumn("sales_ship_date", F.try_to_date(F.col("sls_ship_dt").cast("string"), "yyyyMMdd"))
df2=df2.withColumn("Sales_order_date", F.try_to_date(F.col("sls_order_dt").cast("string"), "yyyyMMdd"))
df2=df2.withColumn("sales_due_date", F.try_to_date(F.col("sls_due_dt").cast("string"), "yyyyMMdd"))
#deleting the table after casting and making new column 
df2 = df2.drop("sls_ship_dt")
df2=df2.drop("sls_order_dt")
df2=df2.drop("sls_due_dt")
#changing data typr from string to integer for quanty and sales and price for analysis
df2 = df2.withColumn("Sales", F.col("sls_sales").cast("integer")).drop("sls_sales")
df2 = df2.withColumn("Quantity", F.col("sls_quantity").cast("integer")).drop("sls_quantity")
df2 = df2.withColumn("Price", F.col("sls_price").cast("integer")).drop("sls_price")
df2=df2.withColumnRenamed("sls_cust_id", "customer_id")
df2=df2.withColumnRenamed("sls_ord_num", "order_number")
df2=df2.withColumnRenamed("sls_prd_key", "product_key")
df2.display()
#Writing in to silver layer
(
    df2.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.Sales_orders")
)


#  Working with CRM customer info table

In [0]:
df3=spark.table("workspace.bronze.crm_cost_info")
df3.describe()

In [0]:
df3.display()

In [0]:
df3=df3.withColumnRenamed("prd_id","product_id")
df3=df3.withColumnRenamed("prd_key","product_key")
df3=df3.withColumnRenamed("prd_nm","product_name")
df3=df3.withColumnRenamed("prd_cost","product_cost")
df3=df3.withColumnRenamed("prd_line","product_line")
df3=df3.withColumnRenamed("prd_start_dt","product_start_date")
df3=df3.withColumnRenamed("prd_end_dt","product_end_date")
df3=df3.withColumn("product_start_date", F.try_to_date(F.col("product_start_date").cast("string"), "yyyy-MM-dd"))
df3=df3.withColumn("product_end_date", F.try_to_date(F.col("product_end_date").cast("string"), "yyyy-MM-dd"))

df3.display()


In [0]:
(
    df3.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.product_info")  
)

In [0]:
%sql
select * from silver.product_info

In [0]:
df4 = spark.table("bronze.px_cat_g1v2")
df4 = df4.withColumnRenamed("CID", "Customer_ID")
df4 = df4.withColumnRenamed("BDATE", "Customer_Birth_Date")
df4 = df4.withColumnRenamed("GEN", "Customer_Gender")
df4.withColumn("Customer_Birth_Date", F.try_to_date(F.col("Customer_Birth_Date").cast("string"), "yyyy-MM-dd"))
df4.withColumn("Customer_Gender", F.when(col("Customer_Gender") == "M", "Male")
                                          .when(col("Customer_Gender") == "F", "Female")
                                          .otherwise("N/A"))
#df4.describe()
df5=spark.table("silver.cust_info")
df5.describe()


In [0]:
df4.write.format("delta").mode("overwrite").saveAsTable("silver.customer_info2")


In [0]:
%sql
drop table if exists workspace.silver.customer_info2

In [0]:
%sql 
select * from workspace.silver.customer_info

This information is already merged with cust_info table. 
df5 = spark.table("workspace.silver.cust_info")

df5 = df5.withColumn("customer_id", F.col("customer_id").cast("string"))

df5.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.cust_info")

UPDATE workspace.silver.customer_info
SET Customer_ID = CONCAT('NASAW000', Customer_ID);


SELECT * FROM workspace.silver.customer_info LIMIT 10;

UPDATE workspace.silver.customer_info
SET Customer_ID = RIGHT(Customer_ID, 5);

select * from workspace.silver.customer_info

alter table workspace.silver.cust_info
add column Customer_Birth_Date date;
    


    
MERGE INTO workspace.silver.cust_info as t1
USING workspace.silver.customer_info as t2
ON t1.customer_id = t2.Customer_ID
WHEN MATCHED THEN UPDATE SET t1.Customer_Birth_Date = t2.Customer_Birth_Date;
  

delete from workspace.silver.cust_info where customer_id is null

drop table if exists workspace.silver.customer_info


In [0]:
%sql
select * from workspace.silver.cust_info

In [0]:
df5=spark.table('workspace.bronze.sales_details')
df5.display()

In [0]:
%sql
select * from workspace.silver.product_info

Renaiming existing talbe in catalog

spark.sql("ALTER TABLE workspace.silver.sales_orders RENAME TO workspace.silver.product_detail")